# IntelliFit Nutrition Model — Kaggle Evaluation
**Model:** `Qwen/Qwen2.5-3B-Instruct` + LoRA adapter (checkpoint-2412, eval_loss=0.2395)  
**Tests:** 3 scenarios × 8 validation dimensions  

**Dataset required:** Upload your adapter + artefacts as a Kaggle dataset.  
Expected path: `/kaggle/input/intellifit-nutrition-adapter/`  
Files needed:
- `adapter_model.safetensors`
- `adapter_config.json`
- `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`, `added_tokens.json`, `merges.txt`, `vocab.json`
- `food_db_halal.json`
- `allergen_taxonomy.json`
- `disease_rules.json`

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
!pip install -q transformers==4.57.3 peft==0.18.1 bitsandbytes accelerate json-repair

In [ ]:
import os, json, re, math, time
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
)
from peft import PeftModel

try:
    from json_repair import repair_json as _repair_json
    JSON_REPAIR = True
except ImportError:
    JSON_REPAIR = False

# ── Paths ───────────────────────────────────────────────────────────────────
ADAPTER_DIR  = "/kaggle/input/intellifit-nutrition-adapter"
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
FOOD_DB_PATH = os.path.join(ADAPTER_DIR, "food_db_halal.json")
ALLERGEN_PATH= os.path.join(ADAPTER_DIR, "allergen_taxonomy.json")
DISEASE_PATH = os.path.join(ADAPTER_DIR, "disease_rules.json")

print("CUDA:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i} =", torch.cuda.get_device_name(i))

In [ ]:
# ── Load artefacts ──────────────────────────────────────────────────────────
with open(FOOD_DB_PATH, encoding="utf-8") as f:
    _food_db = json.load(f)
with open(ALLERGEN_PATH, encoding="utf-8") as f:
    _allergen = json.load(f)
with open(DISEASE_PATH, encoding="utf-8") as f:
    _diseases = json.load(f)

print(f"Food DB      : {len(_food_db):,} entries")
print(f"Allergen DB  : {len(_allergen):,} entries")
print(f"Disease rules: {len(_diseases)} diseases")

In [ ]:
# ── Load model (4-bit QLoRA) ────────────────────────────────────────────────
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map={"" : 0},
    trust_remote_code=True,
    dtype=torch.float16,
)
print(f"  Base model loaded in {time.time()-t0:.1f}s")

print("Loading LoRA adapter...")
t1 = time.time()
_model = PeftModel.from_pretrained(base, ADAPTER_DIR, device_map={"" : 0})
_model.eval()
print(f"  Adapter loaded in {time.time()-t1:.1f}s")

_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
_tokenizer.pad_token = _tokenizer.eos_token

_device = next(_model.parameters()).device
print(f"\nInference device: {_device}")
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"VRAM used: {alloc:.2f} GB allocated  /  {reserved:.2f} GB reserved")
print("Model ready.")

In [ ]:
# ── Stopping criteria: stop when root JSON object closes ───────────────────
class JsonRootClosedCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len: int):
        self._tok = tokenizer
        self._prompt_len = prompt_len
        self._scanned_len = 0
        self._depth = 0
        self._in_str = False
        self._esc = False
        self._started = False

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        new_ids = input_ids[0][self._prompt_len:]
        total_new = len(new_ids)
        if total_new <= self._scanned_len:
            return False
        incremental_text = self._tok.decode(
            new_ids[self._scanned_len:], skip_special_tokens=True)
        self._scanned_len = total_new
        for ch in incremental_text:
            if self._esc:   self._esc = False; continue
            if ch == "\\" and self._in_str: self._esc = True; continue
            if ch == '"':  self._in_str = not self._in_str; continue
            if self._in_str: continue
            if ch == '{':  self._depth += 1; self._started = True
            elif ch == '}':
                self._depth -= 1
                if self._started and self._depth == 0:
                    return True
        return False


# ── Prompt builder ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are IntelliFit Nutrition Coach — an expert Egyptian sports nutritionist. "
    "You create personalised 3-day halal nutrition plans (valid JSON, no markdown). "
    "Each plan has 3 unique days, each with breakfast, lunch, dinner, and one snack. "
    "Plans respect the user's health conditions, allergies, fitness goal, and InBody data. "
    "Output ONLY valid JSON in the exact schema requested. "
    "Use Egyptian food names whenever possible. All food is halal."
)

GOAL_MULTIPLIERS  = {"weight_loss": 0.80, "maintenance": 1.0, "muscle_gain": 1.10, "body_recomposition": 0.95}
ACTIVITY_FACTORS  = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9}

def compute_tdee(gender, age, weight_kg, height_cm, activity_level, goal):
    if gender == "male":
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161
    tdee = bmr * ACTIVITY_FACTORS.get(activity_level, 1.55)
    return int(tdee * GOAL_MULTIPLIERS.get(goal, 1.0))

def build_prompt(gender, age, weight_kg, height_cm, goal, activity_level,
                 health_conditions, allergies, cuisine_preference, daily_kcal):
    conditions_str = ", ".join(health_conditions) if health_conditions else "none"
    allergy_str    = ", ".join(allergies) if allergies else "none"
    disease_notes  = []
    for cond in health_conditions:
        rule = _diseases.get(cond.lower(), {})
        if rule:
            rec   = rule.get("recommended_foods", [])[:3]
            avoid = rule.get("foods_to_avoid", [])[:3]
            if rec:   disease_notes.append(f"Recommended for {cond}: {', '.join(rec)}")
            if avoid: disease_notes.append(f"Avoid for {cond}: {', '.join(avoid)}")
    disease_notes_str = " ".join(disease_notes)
    user_msg = (
        f"Create a 3-day halal nutrition plan for a {age}-year-old {gender}, "
        f"weight {weight_kg} kg, height {height_cm} cm. "
        f"Goal: {goal.replace('_', ' ')}, activity: {activity_level}. "
        f"Health conditions: {conditions_str}. "
        f"Allergies: {allergy_str}. "
        f"Cuisine preference: {cuisine_preference}. "
        f"Daily calorie target: {daily_kcal} kcal. "
        f"Macro targets — protein: 25%, carbs: 50%, fat: 25%. "
        + (disease_notes_str + " " if disease_notes_str else "")
        + "Return only a valid JSON 3-day nutrition plan with breakfast, lunch, dinner, and snack per day."
    )
    return user_msg


# ── JSON extraction with repair fallback ───────────────────────────────────
def extract_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.MULTILINE)
    text = re.sub(r"```\s*$", "", text.strip(), flags=re.MULTILINE).strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object in model output.")
    try:
        return json.loads(text[start:])
    except json.JSONDecodeError:
        pass
    depth, end = 0, -1
    for i, ch in enumerate(text[start:], start):
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: end = i; break
    if end != -1:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    if JSON_REPAIR:
        try:
            r = _repair_json(text[start:], return_objects=True)
            if isinstance(r, dict) and r: return r
        except Exception:
            pass
    raise ValueError("Could not parse JSON from model output.")


def correct_plan_calories(plan: dict) -> dict:
    for day in plan.get("days", []):
        meals = day.get("meals", {})
        actual = sum(
            m.get("total_calories", m.get("estimated_calories", 0))
            for m in (meals.values() if isinstance(meals, dict) else meals)
        )
        if actual > 0:
            day["total_calories"] = actual
    return plan


# ── Main inference function ─────────────────────────────────────────────────
def generate_plan(gender, age, weight_kg, height_cm, goal, activity_level,
                  health_conditions=None, allergies=None,
                  cuisine_preference="egyptian"):
    if health_conditions is None: health_conditions = []
    if allergies is None:         allergies = []

    daily_kcal = compute_tdee(gender, age, weight_kg, height_cm, activity_level, goal)
    user_msg   = build_prompt(gender, age, weight_kg, height_cm, goal, activity_level,
                              health_conditions, allergies, cuisine_preference, daily_kcal)
    messages   = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    prompt = _tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(_device)
    prompt_len = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([JsonRootClosedCriteria(_tokenizer, prompt_len)])

    t0 = time.time()
    with torch.no_grad():
        output_ids = _model.generate(
            **inputs,
            max_new_tokens=4500,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=_tokenizer.eos_token_id,
            pad_token_id=_tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
        )
    gen_ms = int((time.time() - t0) * 1000)

    new_tokens = output_ids[0][prompt_len:]
    print(f"  Generated {len(new_tokens)} tokens in {gen_ms/1000:.1f}s")
    generated = _tokenizer.decode(new_tokens, skip_special_tokens=True)

    plan = extract_json(generated)
    plan = correct_plan_calories(plan)
    plan["_daily_calories"] = daily_kcal
    return {"plan": plan, "daily_calories": daily_kcal, "generation_ms": gen_ms}

print("Inference helpers ready.")

In [ ]:
# ── Evaluation helpers ──────────────────────────────────────────────────────
HALAL_FORBIDDEN = ["pork","bacon","ham","lard","gelatin","alcohol","wine","beer","pig"]
EGYPTIAN_FOODS  = ["ful","foul","baladi","koshari","koshary","ta'ameya","tameya",
                   "feteer","fiteer","molokhia","molokheya","mahshi","rice","lentil",
                   "chickpea","hummus","tahini","eggplant","feta","halloumi",
                   "kofta","kebab","shawarma","hawawshi","semit","aish"]
PASS, FAIL, WARN = "✅ PASS", "❌ FAIL", "⚠️  WARN"

def mifflin_tdee(gender, age, weight_kg, height_cm, activity_level):
    if gender == "male":
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161
    mults = {"sedentary":1.2,"light":1.375,"moderate":1.55,"active":1.725,"very_active":1.9}
    return bmr * mults.get(activity_level, 1.55)

def goal_adjust(tdee, goal):
    return {"weight_loss": tdee*0.80, "maintenance": tdee, "muscle_gain": tdee*1.10}[goal]

def check_plan(name, resp, gender, age, weight_kg, height_cm, goal, activity,
               required_health_conditions=None, forbidden_allergens=None):
    print(f"\n{'='*60}")
    print(f"EVALUATING: {name}")
    print(f"{'='*60}")
    issues, passes = [], []
    plan          = resp.get("plan", {})
    daily_cal_api = resp.get("daily_calories", 0)
    gen_ms        = resp.get("generation_ms", 0)
    print(f"  Generation time : {gen_ms/1000:.1f}s")
    print(f"  API daily kcal  : {daily_cal_api}")

    # 1. Calorie target
    tdee      = mifflin_tdee(gender, age, weight_kg, height_cm, activity)
    expected  = goal_adjust(tdee, goal)
    deviation = abs(daily_cal_api - expected) / expected * 100
    tag = PASS if deviation <= 15 else (WARN if deviation <= 25 else FAIL)
    print(f"\n  [CALORIE TARGET]")
    print(f"    TDEE={tdee:.0f}  expected={expected:.0f}  model={daily_cal_api}  dev={deviation:.1f}%  {tag}")
    (passes if tag == PASS else issues).append("calorie_target" if tag == PASS else f"calorie_target dev {deviation:.1f}%")

    # 2. Structure
    days = plan.get("days", [])
    print(f"\n  [STRUCTURE]  days={len(days)} (expected 3)")
    if len(days) == 3: passes.append("3_days"); print(f"    {PASS} 3 days")
    else:              issues.append(f"expected 3 days got {len(days)}"); print(f"    {FAIL}")
    required_meals = {"breakfast","snack","lunch","dinner"}
    meal_ok = all(required_meals <= set(d.get("meals",{}).keys()) for d in days)
    if meal_ok: passes.append("all_meals"); print(f"    {PASS} breakfast/snack/lunch/dinner every day")
    else:       issues.append("missing meals"); print(f"    {FAIL} some meals missing")

    # 3. Calorie math
    print(f"\n  [CALORIE MATH per day]")
    cal_math_ok = True
    for d in days:
        dt = d.get("total_calories",0)
        ms = sum(m.get("total_calories",0) for m in d.get("meals",{}).values())
        dp = abs(dt - ms) / max(dt,1) * 100
        t2 = PASS if dp<=10 else (WARN if dp<=20 else FAIL)
        print(f"    Day {d.get('day')}: declared={dt}  meal_sum={ms:.0f}  diff={dp:.1f}%  {t2}")
        if dp > 10: cal_math_ok = False; issues.append(f"day{d.get('day')} cal mismatch {dp:.1f}%")
    if cal_math_ok: passes.append("calorie_math")

    # 4. Macro math
    print(f"\n  [MACRO MATH]")
    macro_ok = True
    for d in days:
        m = d.get("macros",{}); p=m.get("protein_g",0); c=m.get("carbs_g",0); f=m.get("fat_g",0)
        mk = p*4+c*4+f*9; dk = d.get("total_calories",0)
        dp = abs(mk-dk)/max(dk,1)*100
        t3 = PASS if dp<=15 else (WARN if dp<=25 else FAIL)
        print(f"    Day {d.get('day')}: P={p}g C={c}g F={f}g → {mk:.0f} kcal  declared={dk}  diff={dp:.1f}%  {t3}")
        if dp > 15: macro_ok = False; issues.append(f"day{d.get('day')} macro mismatch {dp:.1f}%")
    if macro_ok: passes.append("macro_math")

    # 5. Halal
    full_text = json.dumps(plan).lower()
    violations = [w for w in HALAL_FORBIDDEN if w in full_text]
    print(f"\n  [HALAL]")
    if not violations: passes.append("halal"); print(f"    {PASS}")
    else:              issues.append(f"halal:{violations}"); print(f"    {FAIL} {violations}")

    # 6. Egyptian cuisine
    found_eg = [w for w in EGYPTIAN_FOODS if w in full_text]
    print(f"\n  [EGYPTIAN CUISINE]")
    if found_eg: passes.append("egyptian"); print(f"    {PASS} {found_eg[:6]}")
    else:        issues.append("no Egyptian keywords"); print(f"    {FAIL}")

    # 7. Allergen exclusion
    if forbidden_allergens:
        fta  = " ".join(x.lower() for x in plan.get("foods_to_avoid",[]))
        meal_text = json.dumps(plan.get("days",[])).lower()
        print(f"\n  [ALLERGEN — {forbidden_allergens}]")
        in_avoid = any(a in fta for a in forbidden_allergens)
        in_meals = any(a in meal_text for a in forbidden_allergens)
        if in_avoid:  passes.append("allergen_in_avoid"); print(f"    {PASS} in foods_to_avoid")
        else:         issues.append("allergen not in avoid"); print(f"    {WARN}")
        if not in_meals: passes.append("allergen_not_in_meals"); print(f"    {PASS} not in meals")
        else:            issues.append("allergen IN meals"); print(f"    {FAIL} allergen found in meals!")

    # 8. Health conditions
    if required_health_conditions:
        avoid_text = " ".join(x.lower() for x in plan.get("foods_to_avoid",[]))
        print(f"\n  [HEALTH CONDITIONS — {required_health_conditions}]")
        if "diabetes" in required_health_conditions:
            kw = [k for k in ["sugar","candy","sweets","syrup","soda","white bread","cake"] if k in avoid_text]
            print(f"    diabetes avoid kw: {kw}")
            if kw: passes.append("diabetes_rules")
            else:  issues.append("no diabetes avoid items")
        if "hypertension" in required_health_conditions:
            kw = [k for k in ["salt","sodium","pickl","processed","chips"] if k in avoid_text]
            print(f"    hypertension avoid kw: {kw}")
            if kw: passes.append("hypertension_rules")
            else:  issues.append("no hypertension avoid items")

    total = len(passes) + len(issues)
    score = len(passes) / max(total,1) * 100
    print(f"\n  {'─'*50}")
    print(f"  SCORE: {len(passes)}/{total}  →  {score:.0f}%")
    if issues: [print(f"    • {i}") for i in issues]
    return {"name": name, "score": score, "passes": passes, "issues": issues,
            "daily_calories": daily_cal_api, "gen_s": gen_ms/1000}

print("Eval helpers ready.")

In [ ]:
# ── SCENARIO 1: Weight Loss | Male 25yo 90kg 178cm | Moderate ───────────────
print("="*60)
print("SCENARIO 1/3 — Weight Loss | Male 25yo 90kg 178cm | Moderate")
print("="*60)
results = []
try:
    r1 = generate_plan("male", 25, 90.0, 178.0, "weight_loss", "moderate",
                       cuisine_preference="egyptian")
    with open("eval1_result.json","w",encoding="utf-8") as f:
        json.dump(r1, f, ensure_ascii=False, indent=2)
    res = check_plan("WeightLoss/Male", r1, "male", 25, 90, 178, "weight_loss", "moderate")
    results.append(res)
except Exception as e:
    print(f"FATAL: {e}")

In [ ]:
# ── SCENARIO 2: Maintenance | Female 45yo 75kg 162cm | Light | Diabetes+HTN ─
print("="*60)
print("SCENARIO 2/3 — Maintenance | Female 45yo 75kg 162cm | Diabetes+HTN")
print("="*60)
try:
    r2 = generate_plan("female", 45, 75.0, 162.0, "maintenance", "light",
                       health_conditions=["diabetes","hypertension"],
                       cuisine_preference="egyptian")
    with open("eval2_result.json","w",encoding="utf-8") as f:
        json.dump(r2, f, ensure_ascii=False, indent=2)
    res = check_plan("Maintenance/Female/Diabetes+HTN", r2, "female", 45, 75, 162,
                     "maintenance", "light",
                     required_health_conditions=["diabetes","hypertension"])
    results.append(res)
except Exception as e:
    print(f"FATAL: {e}")

In [ ]:
# ── SCENARIO 3: Muscle Gain | Male 28yo 75kg 180cm | Active | Gluten allergy ─
print("="*60)
print("SCENARIO 3/3 — Muscle Gain | Male 28yo 75kg 180cm | Gluten allergy")
print("="*60)
try:
    r3 = generate_plan("male", 28, 75.0, 180.0, "muscle_gain", "active",
                       allergies=["gluten"],
                       cuisine_preference="egyptian")
    with open("eval3_result.json","w",encoding="utf-8") as f:
        json.dump(r3, f, ensure_ascii=False, indent=2)
    res = check_plan("MuscleGain/Male/Gluten", r3, "male", 28, 75, 180,
                     "muscle_gain", "active",
                     forbidden_allergens=["gluten","wheat"])
    results.append(res)
except Exception as e:
    print(f"FATAL: {e}")

In [ ]:
# ── OVERALL ACCURACY REPORT ─────────────────────────────────────────────────
print("\n" + "="*60)
print("OVERALL ACCURACY REPORT")
print("="*60)
if results:
    avg = sum(r["score"] for r in results) / len(results)
    for r in results:
        bar = "█" * int(r["score"]//10) + "░" * (10 - int(r["score"]//10))
        print(f"  {r['name']:<38} {bar}  {r['score']:.0f}%  ({r['gen_s']:.0f}s)")
    print(f"\n  {'─'*50}")
    print(f"  AVERAGE MODEL ACCURACY: {avg:.1f}%")
    if avg >= 80:   print("  ✅ Model is performing well")
    elif avg >= 60: print("  ⚠️  Model needs improvement in some areas")
    else:           print("  ❌ Model has significant accuracy issues")
    print("\n  Result files: eval1_result.json  eval2_result.json  eval3_result.json")
else:
    print("  No results — all scenarios failed.")